# RQ3a — Chain-of-Thought Intervention
Compares CoT accuracy against RQ1 baseline. Does step-by-step reasoning help resist peer pressure?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

RQ3A_DIR = Path("../results_todos/rq3a")
RQ1_DIR  = Path("../results_todos/rq1")

RQ1_CONDITIONS = [
    "all_correct_absent", "all_correct_present",
    "mixed_absent",       "mixed_present",
    "all_wrong_absent",   "all_wrong_present",
]
CONDITION_LABELS = {
    "all_correct_absent":  "All-correct\n(no auth)",
    "all_correct_present": "All-correct\n(auth)",
    "mixed_absent":        "Mixed\n(no auth)",
    "mixed_present":       "Mixed\n(auth)",
    "all_wrong_absent":    "All-wrong\n(no auth)",
    "all_wrong_present":   "All-wrong\n(auth)",
}

def load_combined(results_dir):
    dfs = []
    for csv_path in sorted(results_dir.rglob("*.metrics.csv")):
        if "primevul_dataset" in csv_path.name or "_all_seeds" in csv_path.name:
            continue
        parts = csv_path.parts
        if not any(p.startswith("seed_") for p in parts):
            continue
        df = pd.read_csv(csv_path)
        df = df[df["condition"].isin(RQ1_CONDITIONS + ["ALL"])]
        if not df.empty:
            dfs.append(df)
    all_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    return all_df.groupby(["model", "condition"]).mean(numeric_only=True).reset_index() if not all_df.empty else pd.DataFrame()

cot = load_combined(RQ3A_DIR)
rq1 = load_combined(RQ1_DIR)

MODELS = sorted(cot["model"].unique())
PALETTE = sns.color_palette("tab10", len(MODELS))
MODEL_COLORS = dict(zip(MODELS, PALETTE))

print(f"Models: {MODELS}")
print(f"Conditions: {sorted(cot['condition'].unique())}")


## Per-model tables (CoT)

> **Note on acc_r1:** The CoT Round 1 prompt asks the model to think step-by-step before answering, which genuinely changes initial responses. The `acc_r1` here reflects CoT-prompted Round 1 accuracy — it is not the same as the standard Round 1 baseline in RQ1, even on identical instances.

In [ ]:
display_cols = ["condition", "n", "acc_r1", "acc_r2", "delta_acc",
                "rev_pct", "harm_pct", "ben_pct"]

for model in MODELS:
    df_m = cot[(cot["model"] == model) & (cot["condition"].isin(RQ1_CONDITIONS))].copy()
    present = [c for c in RQ1_CONDITIONS if c in df_m["condition"].values]
    df_m["condition"] = pd.Categorical(df_m["condition"], categories=present, ordered=True)
    df_m = df_m.sort_values("condition")[display_cols].reset_index(drop=True)
    print(f"\n{'─'*60}\n  {model.upper()} — CoT\n{'─'*60}")
    display(df_m.style
        .format({"n": lambda x: f"{int(x)}" if pd.notna(x) and x == int(x) else (f"{x:.2f}" if pd.notna(x) else "N/A"),
                 **{c: "{:.1f}" for c in ["acc_r1","acc_r2","delta_acc","rev_pct","harm_pct","ben_pct"]}}, na_rep="N/A")
        .background_gradient(subset=["delta_acc"], cmap="RdYlGn", vmin=-30, vmax=30)
        .background_gradient(subset=["harm_pct"],  cmap="Reds",   vmin=0,   vmax=100)
        .background_gradient(subset=["ben_pct"],   cmap="Greens", vmin=0,   vmax=100)
        .set_caption(f"{model} CoT"))


## CoT vs Base: Δ Accuracy comparison (grouped bar)

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(5*len(MODELS), 5), sharey=True)
if len(MODELS) == 1:
    axes = [axes]

for ax, model in zip(axes, MODELS):
    for df, label, color in [(rq1, "Base", "#4878d0"), (cot, "CoT", "#ee854a")]:
        df_m = df[(df["model"] == model) & (df["condition"].isin(RQ1_CONDITIONS))].copy()
        p = [c for c in RQ1_CONDITIONS if c in df_m["condition"].values]
        df_m["condition"] = pd.Categorical(df_m["condition"], categories=p, ordered=True)
        df_m = df_m.sort_values("condition")
        x = range(len(df_m))
        offset = -0.175 if label == "Base" else 0.175
        ax.bar([i + offset for i in x], df_m["delta_acc"], 0.35, label=label, color=color, alpha=0.85)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(model, fontsize=12, fontweight="bold")
    ax.set_xticks(list(range(len(p))))
    ax.set_xticklabels([CONDITION_LABELS.get(c, c) for c in p], fontsize=8)
    ax.set_ylabel("\u0394 Accuracy (pp)" if ax == axes[0] else "")
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:+.0f}pp"))

plt.suptitle("Base vs CoT: \u0394 Accuracy by Condition", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(RQ3A_DIR / "rq3a_cot_vs_base.png", dpi=150, bbox_inches="tight")
plt.show()


## Δ Accuracy heatmap — CoT across models × conditions

In [ ]:
df_plot = cot[cot["condition"].isin(RQ1_CONDITIONS)]
pivot = df_plot.pivot_table(index="condition", columns="model", values="delta_acc")
pivot = pivot.reindex([c for c in RQ1_CONDITIONS if c in pivot.index])

fig, ax = plt.subplots(figsize=(max(6, 2.5*len(MODELS)), 5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn", center=0, vmin=-55, vmax=55,
            linewidths=0.5, ax=ax, annot_kws={"size": 10})
ax.set_title("CoT \u0394 Accuracy by Model and Condition", fontsize=13, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_yticklabels([CONDITION_LABELS.get(c, c).replace("\n", " ") for c in pivot.index], rotation=0)
plt.tight_layout()
plt.savefig(RQ3A_DIR / "rq3a_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
